In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline
import menstrualmodel as mm

In [ ]:
# Define constants used in equations.
a_1, a_2 = 0, 0
b_1, b_2 = 0.02, 0.9
s_1, s_2 = 2, 1.5
mu = 0.002
k = 0.000025    # The paper uses 0.00025, but it's hard to determine which solution actually makes sense.
g = 30
c = 0.007
B_1, B_2 = 14, 1
A_1, A_2 = 250000, 75

constants = a_1, a_2, b_1, b_2, s_1, s_2, mu, k, g, c, B_1, B_2, A_1, A_2

# Define other constants.
G0, L0, E0 = 1, 1, 1
n = 2000
t_f = 50

initial_conditions = {
    'GnRH': 1.0,
    'LH': 0.25,
    'Estrogen': 1.0
    }


model = mm.ControlMenstrualModel(initial_hormones=initial_conditions, time_domain=(0, 3*28), resolution=1000)
sol = model.simulate([0, 0, 0])
model.plot(sol, plot_day_14=True, ylim=(0, 12), title="Best Fit Model")
plt.show()

In [ ]:
# Initialize state, costate, and u.
state0 = np.array([G0, L0, E0])
costate0 = np.zeros(2)

u = np.zeros((2, n))
u[0], u[1] = b_1, b_2

max_step = 0.5

epsilon = 0.001
test = epsilon + 1

tls = np.linspace(0, t_f, n)
while(test > epsilon):
    oldu = u.copy()
    u_interpolation = CubicSpline(tls, oldu, axis=1)

    # Solve the state equations forward in time.
    state_solution = solve_ivp(
        model.ode, 
        t_span=(0, t_f), 
        y0=state0, 
        method='RK45',
        t_eval=tls,
        args=(u_interpolation, constants),
        dense_output=True,
        max_step=max_step
        )

    # Solve the costate equations backward in time.
    costate_solution = solve_ivp(
        model.costate_equations,
        t_span=(t_f, 0),
        y0=costate0,
        method='RK45',
        t_eval=tls[::-1],
        args=(u_interpolation, state_solution, constants),
        dense_output=True,
        max_step=max_step
        )

    # Solve for u1 and u2.
    T, V = state_solution.y
    lam1, lam2 = costate_solution.y[:, ::-1]

    u1 = np.clip((1 / (2 * A_1)) * (lam1 * T), a_1, b_1)
    u2 = np.clip((-lam2 / (2 * A_2)) * ((g * V) / (B_2 + V)), a_2, b_2)
    
    # Update control u with u1 and u2.
    u = np.array([u1, u2])

    # Test for convergence
    test = abs(oldu - u).sum()